In [ ]:
!nvidia-smi

Thu Feb 19 13:10:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# 如果文件在 Drive 中，复制到工作目录
!cp /content/drive/MyDrive/AIMS5740/*.py .
!cp /content/drive/MyDrive/AIMS5740/*.yaml .
!mkdir data
!cp /content/drive/MyDrive/AIMS5740/data/* data/.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
mkdir: cannot create directory ‘data’: File exists


In [ ]:
!pip install -q datasets transformers accelerate
!pip install -q "llamafactory[torch,metrics]"
!pip install -q "lm_eval[hf]" tiktoken sentencepiece
!pip install -q matplotlib pyyaml
!pip install -q bitsandbytes

print("✓ 所有依赖安装完成！")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.5/398.5 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install llamafactory

In [ ]:
!llamafactory-cli train train.yaml

2026-02-21 19:12:32.577138: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771701152.599369    5115 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771701152.606889    5115 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771701152.626697    5115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771701152.626736    5115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771701152.626740    5115 computation_placer.cc:177] computation placer alr

In [5]:
!pip -q install -U "lm_eval[hf]" tiktoken sentencepiece

In [9]:
!mkdir output
!cp -r /content/drive/MyDrive/AIMS5740_backup/* output/.

mkdir: cannot create directory ‘output’: File exists


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
SFT_MODEL_DIR = "output/qwen05b_sft_full"
model = AutoModelForCausalLM.from_pretrained(SFT_MODEL_DIR, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_DIR)
print("加载成功！")

加载成功！


In [11]:
!pip install -U lm-eval[hf] tiktoken sentencepiece

import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'  # 缓解显存碎片化

import json
import torch
import gc
from lm_eval import simple_evaluate
from lm_eval.utils import make_table

BASE_MODEL_DIR = "Qwen/Qwen2.5-0.5B"
SFT_MODEL_DIR = "output/qwen05b_sft_full"
OUT_DIR = "eval"
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_ARGS_BASE = f"pretrained={BASE_MODEL_DIR},trust_remote_code=True,dtype=float16"
MODEL_ARGS_SFT = f"pretrained={SFT_MODEL_DIR},trust_remote_code=True,dtype=float16"

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DEFAULT_BATCH_SIZE = 8

TASKS = {
    "mmlu": 5,
    "arc_easy": 0,
    "arc_challenge": 25,      # 25-shot，上下文长，显存需求高
    "hellaswag": 10,
    "winogrande": 5,
    "truthfulqa_mc2": 0,
    "piqa": 0,
    "boolq": 0
}

TASK_BATCH_SIZE_OVERRIDE = {
    "arc_challenge": 4,        # 降低为 4，避免 OOM
}

def evaluate_model(model_args, model_tag):
    print(f"\n评估 {model_tag} 模型")
    all_results = {}

    for task, fewshot in TASKS.items():
        batch = TASK_BATCH_SIZE_OVERRIDE.get(task, DEFAULT_BATCH_SIZE)
        print(f"\n--- 评估任务: {task} (fewshot={fewshot}, batch_size={batch}) ---")

        # 每次任务前清理显存
        gc.collect()
        torch.cuda.empty_cache()

        try:
            result = simple_evaluate(
                model="hf",
                model_args=model_args,
                tasks=[task],
                num_fewshot=fewshot,
                batch_size=batch,
                device=DEVICE,
                apply_chat_template=True,
                limit=None,
            )
            all_results[task] = result['results'][task]
            acc = result['results'][task].get('acc,none', result['results'][task].get('acc', 'N/A'))
            print(f"{task} 完成，准确率: {acc}")
        except Exception as e:
            print(f"{task} 评估失败: {e}")
            all_results[task] = {"error": str(e)}

        # 任务结束后清理
        gc.collect()
        torch.cuda.empty_cache()

    results = {
        'results': all_results,
        'config': {
            'model_args': model_args,
            'batch_size': DEFAULT_BATCH_SIZE,
            'overrides': TASK_BATCH_SIZE_OVERRIDE,
            'device': DEVICE,
            'apply_chat_template': True
        }
    }

    out_path = os.path.join(OUT_DIR, f"results_{model_tag}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"\n所有任务完成，结果已保存至 {out_path}")

    print(f"\n{model_tag} 模型各任务准确率")
    for task_name, task_res in results['results'].items():
        acc = task_res.get('acc,none', task_res.get('acc', 'N/A'))
        print(f"{task_name}: {acc}")

    return results

BASE_RESULTS_PATH = os.path.join(OUT_DIR, "results_base.json")
if os.path.exists(BASE_RESULTS_PATH):
    print("找到已保存的基础模型结果")
    with open(BASE_RESULTS_PATH, "r", encoding="utf-8") as f:
        results_base = json.load(f)
    print("已加载的基础模型准确率：")
    for task_name, task_res in results_base['results'].items():
        acc = task_res.get('acc,none', task_res.get('acc', 'N/A'))
        print(f"{task_name}: {acc}")
else:
    print("未找到基础模型结果，开始评估基础模型")
    results_base = evaluate_model(MODEL_ARGS_BASE, "base")

# sft
gc.collect()
torch.cuda.empty_cache()
results_sft = evaluate_model(MODEL_ARGS_SFT, "sft")

def print_comparison(base_results, sft_results):
    print("\n========== 基础模型 vs SFT 模型对比 ==========")
    print(f"{'Task':<20} {'Base Acc':<12} {'SFT Acc':<12} {'Δ':<8}")
    for task in TASKS.keys():
        base_acc = base_results['results'][task].get('acc,none', base_results['results'][task].get('acc', None))
        sft_acc = sft_results['results'][task].get('acc,none', sft_results['results'][task].get('acc', None))
        if base_acc is not None and sft_acc is not None:
            delta = sft_acc - base_acc
            print(f"{task:<20} {base_acc:<12.4f} {sft_acc:<12.4f} {delta:+.4f}")
        else:
            base_err = base_results['results'][task].get('error', '无数据')
            sft_err = sft_results['results'][task].get('error', '无数据')
            print(f"{task:<20} 错误: base={base_err}, sft={sft_err}")
    print("=============================================")

print_comparison(results_base, results_sft)

找到已保存的基础模型结果，正在加载...
已加载的基础模型准确率：
mmlu: 0.2530978493092152
arc_easy: 0.49242424242424243
arc_challenge: 0.29180887372013653
hellaswag: 0.3836885082652858
winogrande: 0.55327545382794
truthfulqa_mc2: 0.4262526338296755
piqa: 0.6887921653971708
boolq: 0.6318042813455658

========== 正在评估 sft 模型 ==========

--- 评估任务: mmlu (fewshot=5, batch_size=8) ---


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Running loglikelihood requests: 100%|██████████| 56168/56168 [01:51<00:00, 503.40it/s]


✅ mmlu 完成，准确率: 0.4265061956986184

--- 评估任务: arc_easy (fewshot=0, batch_size=8) ---


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

Running loglikelihood requests: 100%|██████████| 3676/3676 [00:14<00:00, 247.57it/s]


✅ piqa 完成，准确率: 0.690424374319913

--- 评估任务: boolq (fewshot=0, batch_size=8) ---


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Running loglikelihood requests: 100%|██████████| 6540/6540 [00:15<00:00, 420.77it/s]


✅ boolq 完成，准确率: 0.6837920489296636

✅ 所有任务完成，结果已保存至 eval/results_sft.json

--- sft 模型各任务准确率 ---
mmlu: 0.4265061956986184
arc_easy: 0.5669191919191919
arc_challenge: 0.31399317406143346
hellaswag: 0.383788090021908
winogrande: 0.56353591160221
truthfulqa_mc2: 0.42643927042545865
piqa: 0.690424374319913
boolq: 0.6837920489296636

========== 基础模型 vs SFT 模型对比 ==========
Task                 Base Acc     SFT Acc      Δ       
mmlu                 0.2531       0.4265       +0.1734
arc_easy             0.4924       0.5669       +0.0745
arc_challenge        0.2918       0.3140       +0.0222
hellaswag            0.3837       0.3838       +0.0001
winogrande           0.5533       0.5635       +0.0103
truthfulqa_mc2       0.4263       0.4264       +0.0002
piqa                 0.6888       0.6904       +0.0016
boolq                0.6318       0.6838       +0.0520


In [ ]:
!cp -r output/ /content/drive/MyDrive/AIMS5740_backup/